# 10. Fixed learned discriminators versus the moving learned ratio

Compare one, two and three fixed coordinates with the moving reference ratio.
Use $z_j=\operatorname{sigmoid}(a_j\log\hat r_j)$, with
$a_{NI}=1$ and $a_{SBI}=a_B=2$. The grids have 36, 1296 and 46656 cells.
The moving ratio uses 36 bins. All binned models use simulation yields;
analytical unbinned is the full-information reference. Learned ratios and finite
MC do not inherit an exact sufficiency guarantee. Inspect sparse-cell counts,
invalid fits and invalid local widths before comparing information fractions.

### Runtime and persistent run

Run from the top in Colab. Setup downloads the pinned tested source into a fresh
temporary directory and clears cached package imports. Keep the existing
`RUN_NAME`, `MODE`, `CONFIG_OVERRIDES` and Unity model directory. Your samples
and models stay on Drive. A GPU is useful for evaluating the large ensembles;
the CDF interpolation and numerical fits also require CPU time.
If authentication is needed, use the `GITHUB_TOKEN` Colab secret or hidden
prompt with repository Contents read access. This update does not require a
new physics run, sample generation, or retraining.

In [ ]:
import os
import sys
from pathlib import Path

RUN_NAME = "paper-distinct-s-v4"
MODE = os.environ.get("POODEMO_MODE", "production")  # production or smoke
CONFIG_OVERRIDES = {}  # e.g. {"quadrature_per_process": 500_000, "epochs": 150}
JAX_BACKEND = os.environ.get("POODEMO_JAX_BACKEND", "auto")  # auto, gpu, or cpu

SOURCE_COMMIT = "18e7fae7714387da787af1cf06695f97fc7d529b"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    requested_backend = JAX_BACKEND.strip().lower()
    if requested_backend not in ("auto", "gpu", "cpu"):
        raise ValueError("JAX_BACKEND must be auto, gpu, or cpu")
    selected_backend = requested_backend
    if requested_backend == "auto":
        try:
            gpu_probe = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                                       capture_output=True, text=True, timeout=10)
            gpu_present = gpu_probe.returncode == 0 and bool(gpu_probe.stdout.strip())
        except (OSError, subprocess.TimeoutExpired):
            gpu_present = False
        selected_backend = "gpu" if gpu_present else "cpu"
    # CUDA-only selection fails visibly if initialization fails. Keep PyTorch's
    # CUDA visibility and avoid JAX reserving most GPU memory before training.
    jax_platform = "cuda" if selected_backend == "gpu" else "cpu"
    os.environ["JAX_PLATFORMS"] = jax_platform
    os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT = Path(os.environ.get("POODEMO_ROOT", f"/content/drive/MyDrive/poodemo/runs/{RUN_NAME}"))
    import tempfile
    # Download afresh even in a reused runtime; do not alter older checkouts.
    REPO_DIR = Path(tempfile.mkdtemp(prefix="poodemo-10-", dir="/content"))
    print(f"Notebook 10 source: {SOURCE_COMMIT} (fresh download)")
    if not (REPO_DIR / "pyproject.toml").exists():
        import getpass
        import io
        import shutil
        import tarfile
        import urllib.error
        import urllib.request

        archive_url = f"https://api.github.com/repos/rafaellopesdesa/poodemo/tarball/{SOURCE_COMMIT}"

        def fetch_source(token=None):
            headers = {"Accept": "application/vnd.github+json", "User-Agent": "poodemo-colab"}
            if token:
                headers["Authorization"] = "Bearer " + token
            request = urllib.request.Request(archive_url, headers=headers)
            with urllib.request.urlopen(request, timeout=180) as response:
                return response.read()

        try:
            archive_bytes = fetch_source()
        except urllib.error.HTTPError as error:
            if error.code not in (401, 403, 404):
                raise RuntimeError(f"Repository download failed (HTTP {error.code}).") from None
            token = None
            try:
                from google.colab import userdata
                token = userdata.get("GITHUB_TOKEN")
            except Exception:
                pass
            if not token:
                token = getpass.getpass("GitHub token with read access to poodemo: ").strip()
            if not token:
                raise RuntimeError("Provide a GitHub token with read access to download the pinned notebook 10 source.")
            try:
                archive_bytes = fetch_source(token)
            except urllib.error.HTTPError as auth_error:
                raise RuntimeError(f"Repository download failed (HTTP {auth_error.code}); check token access.") from None
            finally:
                token = None

        # Validate archive paths and reject links before extracting the source.
        with tempfile.TemporaryDirectory(prefix="poodemo-source-") as temp:
            staging = Path(temp).resolve()

            def safe_destination(name):
                destination = (staging / name).resolve()
                if not destination.is_relative_to(staging):
                    raise RuntimeError("Unsafe path in source archive.")
                return destination

            with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode="r:gz") as archive:
                for entry in archive.getmembers():
                    safe_destination(entry.name)
                    if not (entry.isfile() or entry.isdir()):
                        raise RuntimeError("Links and special files are not supported in the source archive.")
                archive.extractall(staging)
            candidates = [p.parent for p in staging.rglob("pyproject.toml") if (p.parent / "poodemo").is_dir()]
            if len(candidates) != 1:
                raise RuntimeError("The downloaded archive must contain one poodemo repository checkout.")
            shutil.copytree(candidates[0], REPO_DIR, dirs_exist_ok=True)
        del archive_bytes

    if os.environ.get("POODEMO_SKIP_INSTALL") != "1":
        from importlib.metadata import version, PackageNotFoundError
        # JAX discovers every installed plugin, even when selecting CPU.
        # Retain already-matched CUDA 12 packages when rerunning GPU setup.
        remove_plugins = []
        for package in ("jax-cuda12-plugin", "jax-cuda12-pjrt", "jax-cuda13-plugin", "jax-cuda13-pjrt"):
            try:
                installed_version = version(package)
            except PackageNotFoundError:
                continue
            if not (selected_backend == "gpu" and package.startswith("jax-cuda12-") and installed_version == "0.5.3"):
                remove_plugins.append(package)
        if remove_plugins:
            subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", *remove_plugins])
        gpu_requirements = (["jax==0.5.3", "jaxlib==0.5.3",
                             "jax-cuda12-plugin[with-cuda]==0.5.3", "jax-cuda12-pjrt==0.5.3"]
                            if selected_backend == "gpu" else [])
        # Resolve with the base requirements so PyTorch's CUDA dependencies are
        # considered together with JAX's. Avoid blanket --upgrade changes.
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r",
                               str(REPO_DIR / "requirements-colab.txt"), *gpu_requirements])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])

        # Check the live notebook kernel before any expensive network training.
        import jax
        import jaxlib
        import jax.numpy as jnp
        if jax.__version__ != "0.5.3" or jaxlib.__version__ != "0.5.3":
            raise RuntimeError("JAX packages are stale in this kernel. Restart the runtime and rerun setup.")
        if selected_backend == "gpu":
            for package in ("jax-cuda12-plugin", "jax-cuda12-pjrt"):
                if version(package) != "0.5.3":
                    raise RuntimeError(f"{package} must match JAX 0.5.3. Restart the runtime and rerun setup.")
        jax.config.update("jax_platforms", jax_platform)
        jax.config.update("jax_enable_x64", True)
        try:
            jax_devices = jax.devices()
            if not jax_devices or any(device.platform != selected_backend for device in jax_devices):
                raise RuntimeError(f"Expected {selected_backend} devices; found {jax_devices}")
            probe_value, probe_gradient = jax.jit(jax.value_and_grad(lambda x: jnp.sum(x*x)))(
                jnp.asarray([1., 2., 3.], dtype=jnp.float64))
            probe_gradient.block_until_ready()
            if (float(probe_value) != 14. or list(probe_gradient) != [2., 4., 6.]
                    or probe_gradient.dtype != jnp.float64
                    or any(device.platform != selected_backend for device in probe_gradient.devices())):
                raise RuntimeError("JAX double-precision value/gradient check failed")
        except Exception as error:
            raise RuntimeError(
                f"JAX {selected_backend} startup failed. Select a Colab GPU runtime if requesting GPU, "
                "then restart the runtime and rerun setup. Use JAX_BACKEND='cpu' only for an intentional CPU run."
            ) from error
        print(f"JAX {jax.__version__} / jaxlib {jaxlib.__version__}: backend={selected_backend}, {jax_devices}; float64 JIT/gradient OK")

    # A new editable install is not activated in an already-running kernel.
    # Expose the inner package directly, including after a failed import cached
    # the outer /content/poodemo checkout as a namespace package.
    import importlib
    source_root = str(REPO_DIR.resolve())
    if source_root in sys.path:
        sys.path.remove(source_root)
    sys.path.insert(0, source_root)
    importlib.invalidate_caches()
    # Run from the top: subsequent cells recreate objects from this checkout.
    # Clear all poodemo submodules, including old plotting and fit functions.
    for module_name in list(sys.modules):
        if module_name == "poodemo" or module_name.startswith("poodemo."):
            del sys.modules[module_name]

else:
    # Locally: pip install -r requirements-colab.txt && pip install -e .
    ROOT = Path(os.environ.get("POODEMO_ROOT", str(Path.home() / "poodemo-runs" / RUN_NAME)))
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "poodemo" / "pipeline.py").exists():
            sys.path.insert(0, str(candidate))
            break

ROOT.mkdir(parents=True, exist_ok=True)
print(f"Run directory: {ROOT}")
print(f"Mode: {MODE}")

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from poodemo.pipeline import create_run

run = create_run(ROOT, mode=MODE, overrides=CONFIG_OVERRIDES)
plt.rcParams.update({"figure.figsize": (7, 4.5), "axes.grid": True, "grid.alpha": 0.2})
display(pd.Series(run.config, name="configuration"))


import mplhep as hep
hep.style.use("ATLAS")
# ATLAS-like typography/ticks only: no experiment label is added.
plt.rcParams.update({"axes.grid": False, "figure.dpi": 110})

### Reuse notebook 03's filtered density ratios

Run notebook 03 with the transferred Unity models first. Keep the same
`RUN_NAME`, `MODE`, `CONFIG_OVERRIDES`, `UNITY_MODEL_DIR`, and
`ENSEMBLE_FILTER_POLICY` here. The default directory is
`My Drive/poodemo/unity_models/paper-distinct-s-v4-unity100/`; it contains
`bundle.json`, `campaign.json`, and `ratios/` directly inside it.

The next cell verifies the transferred bundle and reuses notebook 03's frozen
retained-member lists for all five ensembles, including the pointwise outlier
rules. It verifies input, policy and artifact hashes without re-screening or
evaluating analytical densities. Changed inputs require a completed matching
filter run in notebook 03 first; no networks are trained here. `RATIO_SOURCE = "drive"` explicitly
selects the original Drive models instead, with the same filtering rules.

All tables and figures go to the selected evaluation's directory, printed
below. The existing Drive samples and frozen preselection remain the inputs. Every coordinate uses the filtered learned ratios. Bin yields
come from labelled held-out simulation events with generated-yield and selection
weights. Only the analytical unbinned curve uses density-oracle evaluations.
The new `practical_v1` output directory preserves notebook 03 and older studies.
Finite MC errors are present in both rates and templates; Asimov data use the
same MC templates, so this tests compression/interpolation, not independent MC
closure. Notebook 11 adds independent simulator experiments.

Run from the top. Setup downloads the pinned source afresh, even in a reused
runtime; existing temporary checkouts are not reused.

In [ ]:
import os
from pathlib import Path
from poodemo.data import save_json
from poodemo.ensemble_bundle import use_bundle, BundleEvaluationRun
from poodemo.member_filter import MemberFilterPolicy, unfiltered_run
from poodemo.practical import (PracticalRun, use_practical_run,
                               reuse_filtered_ensembles as filter_ratio_ensembles)

RATIO_SOURCE = "unity"  # "drive" explicitly selects the original saved models.
UNITY_MODEL_DIR = Path(os.environ.get(
    "POODEMO_UNITY_MODEL_DIR",
    "/content/drive/MyDrive/poodemo/unity_models/paper-distinct-s-v4-unity100",
))
ENSEMBLE_FILTER_POLICY = MemberFilterPolicy(
    catastrophic_peer_factor=100.0,
)  # Match notebook 03, including any custom policy settings.

# Unwrap a previous selection so rerunning this cell never nests evaluations.
source_run = run.base_run if isinstance(run, PracticalRun) else run
source_run = unfiltered_run(source_run)
source_run = source_run.base_run if isinstance(source_run, BundleEvaluationRun) else source_run
if RATIO_SOURCE == "unity":
    if not (UNITY_MODEL_DIR / "bundle.json").is_file():
        raise FileNotFoundError(
            f"Copy the complete Unity model bundle to {UNITY_MODEL_DIR}. "
            "Expected bundle.json, campaign.json and ratios/ directly inside it. "
            "Use the same directory as notebook 03; see unity/README.md.")
    ensemble_run = use_bundle(source_run, UNITY_MODEL_DIR)
elif RATIO_SOURCE == "drive":
    ensemble_run = source_run
else:
    raise ValueError("RATIO_SOURCE must be 'drive' or 'unity'")
run = filter_ratio_ensembles(ensemble_run, policy=ENSEMBLE_FILTER_POLICY)
filter_summary = pd.read_csv(run.path("results", "ensemble_filter", "summary.csv"))
display(filter_summary)
print("Ratio source:", RATIO_SOURCE)
print("Retained ratio models:", run.path("models", "ratios"))
print("Evaluation outputs:", run.evaluation_root)
save_json(run.path("results", "reference_ratio_model_source.json"), {
    "ratio_source": RATIO_SOURCE,
    "bundle_directory": str(ensemble_run.bundle) if RATIO_SOURCE == "unity" else None,
    "campaign_id": ensemble_run.bundle_manifest["campaign_id"] if RATIO_SOURCE == "unity" else None,
    "filter_identity": run.filter_manifest["identity"],
    "filter_policy": run.filter_manifest["inputs"]["policy"],
    "evaluation_root": str(run.evaluation_root),
})
# Cap selected integration events per source; all events estimate acceptance.
SIMULATION_EVENTS_PER_SAMPLE = 250_000
_filtered_run = run
run = use_practical_run(run).with_config(simulation_per_sample=SIMULATION_EVENTS_PER_SAMPLE)
save_json(run.path("results", "reference_ratio_model_source.json"),
          __import__("json").loads(_filtered_run.path("results", "reference_ratio_model_source.json").read_text()))
print("Practical study outputs:", run.evaluation_root)

### Learned observables and simulation templates

All event coordinates use the same **filtered Unity ensembles** as notebook 03.
The moving observable uses $\hat p_\eta/\hat p_1$, with power $100$ and **no floor**.
Fixed process axes use the learned ratios to $S$. Rates and ratio normalizers
are frozen on held-out simulation before fits or toys; the common unknown $S$
density cancels. At $\eta=1$ the moving observable is exactly $1/2$.

Templates come from labelled integration events for $S$, $SBI$, $B$, $NI$ and
its two variations. For sample $j$, each retained shape event receives weight
$\nu_j/n_j$, where $\nu_j=\lambda_j N_{j,\mathrm{selected}}/N_{j,\mathrm{generated}}$.
`SIMULATION_EVENTS_PER_SAMPLE` caps selected events used per source (default
250,000); all selected events estimate the acceptance. Increase the cap to
study MC convergence without regenerating or training. Predictions are cached
with sample hashes, filter identity and normalizers.

Only **analytical unbinned** is an oracle benchmark. The analytical toy model
still defines the simulator. Analytical PDFs are not used to define observables,
MC bin yields, or learned unbinned likelihoods. Binned Asimov observations use
these same nominal MC templates: closure at truth does not prove independence
from MC errors. Notebook 11 supplies fresh paired simulator experiments.

NI is morphed **after binning**. Positive anchors retain the existing exponential
polynomial rate/shape morph. A bin with a zero NI anchor uses linear interpolation
of its shape between $-1,0,+1$, holding the boundary shape beyond that range;
total rates still use the exponential polynomial morph. This explicit sparse-MC
convention adds no pseudo-events or yield floor. Inspect the support diagnostics
and repeat with more MC when affected bins matter. Nonregular local widths and
negative signed physical predictions remain invalid and visible.

New tables, figures and toy checkpoints go under **`practical_v1`** inside the
selected ensemble/filter evaluation. Historical outputs, including notebook 12,
remain available. Run **08 before 09**; 10 and 11 can run after selecting the same
completed 01–03 run. No retraining or transfer of event weights is needed.

In [ ]:
from poodemo.discriminator_study import run_discriminator_study, plot_discriminator_study

N_BINS = 36  # Per coordinate: 36, 36**2, and 36**3 cells.
POWERS = {"NI": 1.0, "SBI": 2.0, "B": 2.0}
REFERENCE_RATIO_A = 100.0
REFERENCE_RATIO_ETA_MIN = 0.0  # No floor.
OUTPUT_TAG = "10_learned_mc_36_no_floor"  # Preserve the previous floor-0.1 study.
AXIS_SOURCE = "learned"  # Filtered ratios; templates use simulation.
SYSTEMATICS = (False, True)  # Compare both fixed and profiled NI.
MORPH_ORDER = "after"  # Interpolate the binned NI templates, as in 09.
MU_GRID = None  # None uses the saved run scan; e.g. np.linspace(0.02, 2., 101).

print(f"Fixed grids: {N_BINS}, {N_BINS**2}, {N_BINS**3} cells")
print("Process powers:", POWERS, "| physical-reference power:", REFERENCE_RATIO_A)
print("Moving numerator floor:", REFERENCE_RATIO_ETA_MIN)
print("Observable axes:", AXIS_SOURCE, "| Asimov truth:", run.config["asimov_mu"])

In [ ]:
study = run_discriminator_study(
    run, n_bins=N_BINS, powers=POWERS, reference_power=REFERENCE_RATIO_A,
    ratio_source=AXIS_SOURCE, systematics=SYSTEMATICS,
    scan_grid=MU_GRID, morph_order=MORPH_ORDER,
    reference_eta_floor=REFERENCE_RATIO_ETA_MIN, output_tag=OUTPUT_TAG,
)
display(study["axis_summary"])
display(study["occupancy"])
print("Full settings:")
display(pd.Series(study["config"]))

In [ ]:
FIGURE_DIR = run.path("plots") / OUTPUT_TAG
figures = plot_discriminator_study(study, output=FIGURE_DIR)
for name, figure in figures.items():
    print(name)
    display(figure)
    plt.close(figure)
print("Figures:", FIGURE_DIR)
print("Tables:", run.path("results") / OUTPUT_TAG)

In [ ]:
display(study["information"])
scans = study["scans"]
if "valid" in scans.columns:
    invalid = scans.loc[~scans["valid"].fillna(False)]
    if len(invalid):
        print("Inspect these unsuccessful fits before interpreting the curves:")
        display(invalid)
display(scans.head(15))